# Basic model introduction

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/01-basic-model` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Build and run a simple SIR model: starting population 1000 with 10 infectious,
infection / recovery / infection-death flows, timespan 0–20 days.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    ExitFlow,
    FlowModel,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"


In [ ]:
state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)

model = FlowModel(pmap)
mixing = MixingMatrix(pop, [[1.0]], check_reciprocal=False)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=mixing.prop,
            mixing=mixing,
            kind="frequency",
            contact_rate=1.0,
        ),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
model.add_flow(ExitFlow("infection_death", state["I"], 0.05))
model.set_initial_population({state["S"]: 990.0, state["I"]: 10.0})

cm = model.compile()
assert float(np.asarray(cm.initial_state({}).data).sum()) == 1000.0
plan = SavePlan(requests={"comp": SaveRequest(Compartments())})
res = cm.run({}, t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
frame = res["comp"].to_pandas()
assert frame.shape[0] > 10
assert float(frame.iloc[0].sum()) == 1000.0
frame.plot(title="Basic SIR with infection deaths")


## Step by step

Create a map whose disease states are an ordinary `Property`, attach infection
via `FlowModel` + `ForceOfInfection`, and declare the initial population instead of building `y0` by
hand.


In [ ]:
assert pmap.size == 3
assert pmap.labels() == ("state=S_pop=all", "state=I_pop=all", "state=R_pop=all")
y0 = cm.initial_state({})
assert float(np.asarray(y0.data)[pmap.select(state["I"])][0]) == 10.0
assert float(np.asarray(y0.data)[pmap.select(state["S"])][0]) == 990.0
